In [ ]:
import pandas as pd
import numpy as np


In [ ]:
from google.colab import files
uploaded = files.upload()


Saving combined_final_with_lags.csv to combined_final_with_lags.csv


In [ ]:
df = pd.read_csv("combined_final_with_lags.csv")

df = df.sort_values(
    by=['City_encoded', 'Year', 'Month']
).reset_index(drop=True)


**PAST-ONLY ROLLING**

In [ ]:
# AQI rolling
df['AQI_roll3_past'] = (
    df.groupby('City_encoded')['avg_AQI']
      .shift(1)
      .rolling(3)
      .mean()
)

df['AQI_roll6_past'] = (
    df.groupby('City_encoded')['avg_AQI']
      .shift(1)
      .rolling(6)
      .mean()
)

# Noise rolling (past only)
df['Day_roll3_past'] = (
    df.groupby('City_encoded')['Day']
      .shift(1)
      .rolling(3)
      .mean()
)

df['Night_roll3_past'] = (
    df.groupby('City_encoded')['Night']
      .shift(1)
      .rolling(3)
      .mean()
)

df = df.dropna().reset_index(drop=True)


**SAFE TIME-BASED SPLIT**

In [ ]:
def time_split_safe(df, target):
    years = sorted(df['Year'].unique())

    train_years = years[:-1]
    test_years  = [years[-1]]

    train = df[df['Year'].isin(train_years)]
    test  = df[df['Year'].isin(test_years)]

    X_train = train.drop(columns=[target])
    y_train = train[target]

    X_test  = test.drop(columns=[target])
    y_test  = test[target]

    return X_train, y_train, X_test, y_test


**METRICS**

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_percentage_error

def evaluate(y_true, y_pred, name):
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    print(f"{name} → R2: {r2:.3f}, MAPE: {mape:.2f}%")
    return r2, mape


**INSTALL XGBOOST**

In [ ]:
!pip install xgboost


In [ ]:
from xgboost import XGBRegressor


**XGBOOST — AQI FORECASTING**

Leak-free features

In [ ]:
aqi_features = [
    'Year', 'Month', 'City_encoded',
    'avg_AQI_lag1', 'avg_AQI_lag2', 'avg_AQI_lag3',
    'AQI_roll3_past', 'AQI_roll6_past'
]

aqi_df = df[aqi_features + ['avg_AQI']]


Train / Test split

In [ ]:
X_train, y_train, X_test, y_test = time_split_safe(
    aqi_df, 'avg_AQI'
)


Tuned XGBoost model (stable & strong)

In [ ]:
xgb_aqi = XGBRegressor(
    n_estimators=800,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_alpha=0.2,
    reg_lambda=1.0,
    objective='reg:squarederror',
    random_state=42
)

xgb_aqi.fit(X_train, y_train)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.9, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.03, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=800,
             n_jobs=None, num_parallel_tree=None, ...)

Evaluate

In [ ]:
aqi_preds = xgb_aqi.predict(X_test)
evaluate(y_test, aqi_preds, "XGBoost AQI Forecast")


XGBoost AQI Forecast → R2: 0.953, MAPE: 7.08%


(0.9528992704100152, 7.082413882858075)

**XGBOOST — DAY NOISE FORECAST**

Features (non-linear friendly)

In [ ]:
day_features = [
    'Year', 'Month', 'City_encoded',
    'Day_lag1', 'Day_lag2', 'Day_lag3',
    'avg_AQI_lag1',
    'Day_roll3_past'
]

day_df = df[day_features + ['Day']]


Train / Test split

In [ ]:
X_train, y_train, X_test, y_test = time_split_safe(
    day_df, 'Day'
)


Tuned XGBoost (noise is harder → shallower trees)

In [ ]:
xgb_day = XGBRegressor(
    n_estimators=600,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.3,
    reg_lambda=1.0,
    objective='reg:squarederror',
    random_state=42
)

xgb_day.fit(X_train, y_train)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.85, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=600,
             n_jobs=None, num_parallel_tree=None, ...)

Evaluate

In [ ]:
day_preds = xgb_day.predict(X_test)
evaluate(y_test, day_preds, "XGBoost Day Noise Forecast")


XGBoost Day Noise Forecast → R2: 0.007, MAPE: 8.62%


(0.007224178278273152, 8.624948417989103)

**XGBOOST — NIGHT NOISE FORECAST**

Features

In [ ]:
night_features = [
    'Year', 'Month', 'City_encoded',
    'Night_lag1', 'Night_lag2',
    'avg_AQI_lag1',
    'Night_roll3_past'
]

night_df = df[night_features + ['Night']]


Train / Test split

In [ ]:
X_train, y_train, X_test, y_test = time_split_safe(
    night_df, 'Night'
)


Tuned XGBoost

In [ ]:
xgb_night = XGBRegressor(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.3,
    reg_lambda=1.0,
    objective='reg:squarederror',
    random_state=42
)

xgb_night.fit(X_train, y_train)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.85, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=None, num_parallel_tree=None, ...)

Evaluation

In [ ]:
night_preds = xgb_night.predict(X_test)
evaluate(y_test, night_preds, "XGBoost Night Noise Forecast")


XGBoost Night Noise Forecast → R2: -0.030, MAPE: 10.47%


(-0.029825256590593208, 10.46741536102305)